# KWISMO — Entrainement du Modele B (NLP, AfroXLMR + LoRA)

A executer sur Google Colab (Runtime > Change runtime type > GPU).
Ce notebook clone le depot, installe les dependances de `kwismo-ai/`, puis importe le meme code source (`src/`) que le service en local.

## 1. Verifier le GPU

In [ ]:
!nvidia-smi

## 2. Cloner le depot

In [ ]:
!git clone https://github.com/newtonachonduh46/kwismo.git
%cd kwismo/kwismo-ai

## 3. Installer les dependances

Colab fournit deja un torch avec CUDA : on installe le reste de `requirements.txt` sans le reinstaller pour ne pas perdre le support GPU.

In [ ]:
!grep -v '^torch' requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt

import torch
print("CUDA disponible :", torch.cuda.is_available())

## 4. Configuration (.env)

`.env` n'est pas versionne : on le recree ici pour cette session Colab (les valeurs par defaut de `src/config.py` conviennent dans la plupart des cas).

In [ ]:
%%writefile .env
MODEL_DIR="./models"
HF_MODEL_NAME="Davlan/afro-xlmr-base"
MODEL_B_VERSION="v1"


## 5. Charger les donnees annotees

Deposer le jeu de donnees annote dans `data/raw/` (upload manuel, ou `data/raw/` monte depuis Google Drive) avant cette etape.

In [ ]:
from src.data.collect import load_raw_reports
from pathlib import Path

# df = load_raw_reports(Path("data/raw/messages_annotes.csv"))
# df.head()

## 6. Entrainer (fine-tuning LoRA)

In [ ]:
from src.models.model_b.train import load_base_model, apply_lora, save

tokenizer, model = load_base_model()
model = apply_lora(model)

# TODO : boucle d'entrainement (Trainer HuggingFace) sur df une fois charge.

save(version="v1")

## 7. Publier l'artefact (retour vers Git)

Un adaptateur LoRA est petit (quelques Mo) : il peut etre commite directement. Necessite un jeton GitHub avec acces en ecriture (Colab > icone cle "Secrets", ou coller le jeton ci-dessous).

In [ ]:
!git config user.email "you@example.com"
!git config user.name "Ton Nom"
!git add models/
!git commit -m "Entrainement Modele B vX (Colab)"
# !git push https://<TOKEN>@github.com/newtonachonduh46/kwismo.git main